In [2]:
import json

with open("tickets_classification_eng.json", "r", encoding="utf-8") as f:
    data = json.load(f)
import pandas as pd
from pandas import json_normalize
df = pd.read_json("tickets_classification_eng.json", lines=True)
# Show first entry
print(json.dumps(data[0], indent=2))


{
  "_index": "complaint-public-v2",
  "_type": "complaint",
  "_id": "3211475",
  "_score": 0.0,
  "_source": {
    "tags": null,
    "zip_code": "90301",
    "complaint_id": "3211475",
    "issue": "Attempts to collect debt not owed",
    "date_received": "2019-04-13T12:00:00-05:00",
    "state": "CA",
    "consumer_disputed": "N/A",
    "product": "Debt collection",
    "company_response": "Closed with explanation",
    "company": "JPMORGAN CHASE & CO.",
    "submitted_via": "Web",
    "date_sent_to_company": "2019-04-13T12:00:00-05:00",
    "company_public_response": null,
    "sub_product": "Credit card debt",
    "timely": "Yes",
    "complaint_what_happened": "",
    "sub_issue": "Debt is not yours",
    "consumer_consent_provided": "Consent not provided"
  }
}


 we cans e all the files or columns of the jsonfile the it doesnt seem to be sensible information here about the clients, all information appear to be about the log entry.

In [3]:
# Cell 1: Imports
import json
import pandas as pd
from collections import Counter
import re
# Cell 2: Load JSON or JSONL file
def load_json_file(path: str) -> pd.DataFrame:
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    else:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return pd.json_normalize(data)


In [ ]:
# Cell 3: Detect likely text fields
def is_text_series(series: pd.Series, min_length=30, min_non_null=10) -> bool:
    """this functions decide if a  pd. series is like ly to contain long text,
    if its ab ibject and has at least 10 entries that  are long text then is natural text pandas series"""
    if series.dtype != object:
        return False
    non_null = series.dropna()
    long_texts = non_null[non_null.str.len() > min_length]
    return len(long_texts) >= min_non_null


In [ ]:
# Cell 4: Analyze a single text field
def analyze_text_field(series: pd.Series, field_name: str):
    """this function is used to analize the fields or columns that has alreally been  clasified as  long natural text series
    """
    non_null = series.dropna()
    print(f"\n🔎 Field: {field_name}")
    print(f"Entries: {len(non_null)}")

    print("\n📌 Example texts:")
    for i, example in enumerate(non_null.sample(3)):
        shortened = example.replace("\n", " ")[:120]
        if len(example) > 120:
            shortened += "..."
        print(f"{i+1}. {shortened}")

    print("\n📊 Top words:")
    text_block = " ".join(non_null).lower()
    tokens = re.findall(r"\b[a-z]{3,}\b", text_block)
    common_words = Counter(tokens).most_common(10)
    for word, count in common_words:
        print(f"- {word}: {count}")


In [6]:
# Cell 5: Run on a file
def explore_text_fields(path: str):
    df = load_json_file(path)
    print(f"✅ Loaded file with shape {df.shape}")

    text_cols = [col for col in df.columns if is_text_series(df[col])]

    if not text_cols:
        print("❌ No likely text fields found.")
        return

    for col in text_cols:
        analyze_text_field(df[col], col)


In [8]:
# Cell 6: Run the analysis
# Replace with the path to your JSON file
file_path = "tickets_classification_eng.json"  # or "your_file.jsonl"
explore_text_fields(file_path)


✅ Loaded file with shape (78313, 22)

🔎 Field: _source.issue
Entries: 78313

📌 Example texts:
1. Attempts to collect debt not owed
2. Written notification about debt
3. Other features, terms, or problems

📊 Top words:
- account: 22743
- loan: 17132
- your: 11818
- collection: 9933
- modification: 9743
- foreclosure: 9743
- problem: 9314
- managing: 8260
- closing: 7908
- payments: 6819

🔎 Field: _source.product
Entries: 78313

📌 Example texts:
1. Debt collection
2. Debt collection
3. Credit card or prepaid card

📊 Top words:
- card: 31999
- credit: 29982
- mortgage: 22725
- account: 21963
- checking: 12147
- savings: 12147
- service: 11376
- prepaid: 10829
- bank: 9816
- consumer: 5339

🔎 Field: _source.company_response
Entries: 78313

📌 Example texts:
1. Closed with explanation
2. Closed with explanation
3. Closed with explanation

📊 Top words:
- closed: 78192
- with: 75555
- explanation: 60230
- relief: 17334
- monetary: 14512
- non: 4383
- without: 2009
- progress: 119
- untimely: 2

all  text  columns are  describe above, the sensible information aparently was already  take out using "xxxx" patterns to replace it.